# Terceiro Trabalho — Reconhecimento de Padrões

Notebook do **terceiro** trabalho: classificação de risco de crédito com o banco **Statlog (German Credit Data)**.

Nesta primeira etapa tratamos apenas do **pré-processamento**, seguindo os tópicos da aula (*Pré-processamento e Avaliação de Classificadores*):

1. Caracterização do conjunto de dados ($n$ instâncias, $l$ parâmetros)
2. Limpeza (valores faltantes e inconsistências)
3. Análise do **desbalanceamento** entre classes
4. Codificação de atributos categóricos
5. **Normalização** (padronização $z$-score)

Os classificadores (MQ, kNN, MLP, SVM, AdaBoost, Random Forest) e a **validação cruzada** serão adicionados em etapas posteriores.

**Dependências:** `numpy`, `pandas`, `matplotlib`, `scikit-learn`


## 1 — Configuração e caminhos

Carregamos a versão **original** do banco (`german.data`), com atributos categóricos simbólicos e numéricos, para aplicar o pré-processamento de forma explícita. A documentação local está em `documents/trabalho3/german_credit_data/german.doc`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Caminhos (notebook em Bayes/)
ROOT = Path("..").resolve()
DATA_PATH = ROOT / "documents" / "trabalho3" / "german_credit_data" / "german.data"
FIG_DIR = ROOT / "documents" / "trabalho3" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42


## 2 — Carregamento dos dados

O arquivo `german.data` contém **1000** linhas. Cada linha descreve um pedido de crédito com **20 atributos** e o rótulo da classe no final:

| Classe | Significado |
|--------|-------------|
| 1 | *Good* — bom pagador |
| 2 | *Bad* — mau pagador |

Conforme a caracterização da aula, o conjunto pode ser visto como uma matriz $\mathbf{X} \in \mathbb{R}^{n \times l}$, com $n$ instâncias e $l$ parâmetros.


In [ ]:
# Nomes das colunas (atributos 1–20 + classe), conforme german.doc / UCI
COLUMN_NAMES = [
    "checking_status",      # 1  - qualitativo
    "duration",             # 2  - meses
    "credit_history",       # 3  - qualitativo
    "purpose",              # 4  - qualitativo
    "credit_amount",        # 5  - numérico
    "savings_status",       # 6  - qualitativo
    "employment",           # 7  - qualitativo
    "installment_rate",     # 8  - % da renda
    "personal_status",      # 9  - qualitativo
    "other_parties",        # 10 - qualitativo
    "residence_since",      # 11 - anos
    "property_magnitude",   # 12 - qualitativo
    "age",                  # 13 - anos
    "other_payment_plans",  # 14 - qualitativo
    "housing",              # 15 - qualitativo
    "existing_credits",     # 16 - quantidade
    "job",                  # 17 - qualitativo
    "num_dependents",       # 18 - quantidade
    "own_telephone",        # 19 - qualitativo
    "foreign_worker",       # 20 - qualitativo
    "class",                # rótulo: 1=good, 2=bad
]

CATEGORICAL_COLS = [
    "checking_status", "credit_history", "purpose", "savings_status",
    "employment", "personal_status", "other_parties", "property_magnitude",
    "other_payment_plans", "housing", "job", "own_telephone", "foreign_worker",
]
NUMERIC_COLS = [
    "duration", "credit_amount", "installment_rate", "residence_since",
    "age", "existing_credits", "num_dependents",
]

df = pd.read_csv(DATA_PATH, sep=r"\s+", header=None, names=COLUMN_NAMES)
df.head()


## 3 — Caracterização: dimensão e tipos

Verificamos $n$, $l$ e a relação $n/l$, relevante para a **maldição da dimensionalidade** discutida em aula: quanto maior $l$, mais instâncias tendem a ser necessárias para generalizar bem.


In [ ]:
n_instances, n_cols = df.shape
n_params = n_cols - 1  # exclui a classe

print(f"n (instâncias)     = {n_instances}")
print(f"l (parâmetros)     = {n_params}")
print(f"relação n/l        = {n_instances / n_params:.1f}")
print()
print("Tipos inferidos:")
print(df.dtypes)
print()
print("Estatísticas descritivas (colunas numéricas):")
df[NUMERIC_COLS].describe().round(2)


## 4 — Limpeza: dados faltantes

A aula lista três abordagens principais para dados incompletos: remover instâncias, imputar (média/moda/mediana) ou usar um valor simbólico de *desconhecido*. Verificamos se há valores ausentes neste banco.


In [ ]:
missing = df.isna().sum()
print("Valores ausentes por coluna:")
print(missing[missing > 0] if missing.any() else "Nenhum valor ausente.")

# Alguns códigos categóricos usam '?' em outras bases UCI; conferimos
question_marks = (df.astype(str) == "?").sum()
print()
print("Marcadores '?' por coluna:")
print(question_marks[question_marks > 0] if question_marks.any() else "Nenhum marcador '?' encontrado.")


## 5 — Limpeza: dados inconsistentes

Dados inconsistentes violam relações esperadas entre atributos. Em bases pequenas como esta ($n=1000$), a verificação pode ser manual ou por regras simples. Aqui fazemos checagens básicas de domínio nas colunas numéricas.


In [ ]:
checks = {
    "duration > 0": (df["duration"] > 0).all(),
    "credit_amount > 0": (df["credit_amount"] > 0).all(),
    "installment_rate em [1, 4]": df["installment_rate"].between(1, 4).all(),
    "age >= 18": (df["age"] >= 18).all(),
    "existing_credits >= 1": (df["existing_credits"] >= 1).all(),
    "num_dependents >= 1": (df["num_dependents"] >= 1).all(),
    "class em {1, 2}": df["class"].isin([1, 2]).all(),
}

for rule, ok in checks.items():
    status = "OK" if ok else "FALHOU"
    print(f"[{status}] {rule}")

print()
print("Valores únicos da classe:", sorted(df["class"].unique()))


## 6 — Desbalanceamento de classes

Em problemas reais as classes costumam ter tamanhos diferentes. O German Credit Data é **desbalanceado** (~70% *good*, ~30% *bad*). A aula menciona reamostragem, custos por classe e métricas adequadas; por ora **apenas analisamos** a distribuição — a estratégia de compensação será definida na etapa de avaliação.


In [ ]:
class_counts = df["class"].value_counts().sort_index()
class_pct = (class_counts / len(df) * 100).round(1)

summary = pd.DataFrame({
    "classe": ["1 — good", "2 — bad"],
    "contagem": class_counts.values,
    "percentual (%)": class_pct.values,
})
summary


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
labels = ["Good (1)", "Bad (2)"]
colors = ["#4c72b0", "#c44e52"]
ax.bar(labels, class_counts.values, color=colors, edgecolor="black", linewidth=0.6)
for i, (cnt, pct) in enumerate(zip(class_counts.values, class_pct.values)):
    ax.text(i, cnt + 8, f"{cnt}\n({pct}%)", ha="center", fontsize=10)
ax.set_ylabel("Número de instâncias")
ax.set_title("Distribuição das classes — German Credit Data")
ax.set_ylim(0, class_counts.max() * 1.15)
fig.tight_layout()

fig_path = FIG_DIR / "preproc_distribuicao_classes.png"
fig.savefig(fig_path, dpi=150)
plt.show()
print(f"Figura salva em: {fig_path}")


## 7 — Separação de atributos e rótulos

Separamos a matriz de parâmetros $\mathbf{X}$ e o vetor de classes $\mathbf{y}$.

Para os classificadores do `scikit-learn`, mapeamos **good $\rightarrow 0$** e **bad $\rightarrow 1$** (convenção binária). O rótulo original permanece em `y_raw`.


In [ ]:
X_raw = df.drop(columns="class")
y_raw = df["class"].copy()

# 1=good -> 0, 2=bad -> 1
y = (y_raw - 1).astype(int)

print("X_raw:", X_raw.shape)
print("y:", y.shape)
print("Distribuição após mapeamento (0=good, 1=bad):")
print(y.value_counts().sort_index())


## 8 — Codificação de variáveis categóricas

Dos 20 atributos de entrada, **13 são categóricos** e **7 são numéricos** (`german.doc`). Algoritmos como kNN, SVM e MLP exigem representação numérica.

Aplicamos **one-hot encoding** às colunas categóricas com `ColumnTransformer`, mantendo as numéricas sem transformação neste passo (a normalização vem a seguir).


In [ ]:
preprocess_encode = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_COLS),
        ("num", "passthrough", NUMERIC_COLS),
    ],
    verbose_feature_names_out=False,
)

X_encoded = preprocess_encode.fit_transform(X_raw)
feature_names = preprocess_encode.get_feature_names_out()

print(f"Dimensão após codificação: {X_encoded.shape[1]} parâmetros")
print(f"(original l={len(NUMERIC_COLS) + len(CATEGORICAL_COLS)} -> expandido por one-hot)")
print()
print("Primeiros 10 nomes de atributos:", list(feature_names[:10]))


## 9 — Normalização (padronização $z$-score)

Conforme a aula, parâmetros em escalas diferentes prejudicam classificadores sensíveis à distância (kNN) e à otimização (SVM, MLP). A padronização usada é

$$x'_j = \frac{x_j - \mu_j}{\sigma_j}$$

onde $\mu_j$ e $\sigma_j$ são média e desvio padrão do $j$-ésimo parâmetro.

**Nota:** em um pipeline com validação cruzada, o `StandardScaler` deve ser ajustado **apenas no fold de treino** para evitar vazamento de informação. Aqui, na etapa exploratória, ajustamos em todo o conjunto para inspecionar o efeito.


In [ ]:
# Estatísticas antes da normalização (colunas numéricas no array codificado)
num_idx = [list(feature_names).index(c) for c in NUMERIC_COLS]
X_num_before = X_encoded[:, num_idx]

before = pd.DataFrame(X_num_before, columns=NUMERIC_COLS)
print("Antes — média e desvio padrão (atributos numéricos):")
print(before.agg(["mean", "std"]).round(2))


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_encoded)

X_num_after = X_scaled[:, num_idx]
after = pd.DataFrame(X_num_after, columns=NUMERIC_COLS)
print("Depois — média e desvio padrão (atributos numéricos):")
print(after.agg(["mean", "std"]).round(2))


## 10 — Resumo do pré-processamento

Ao final desta etapa dispomos de:

| Objeto | Descrição |
|--------|-----------|
| `X_raw` | DataFrame original ($n \times 20$) |
| `X_encoded` | Matriz após one-hot ($n \times l'$) |
| `X_scaled` | Matriz padronizada (entrada para kNN, SVM, MLP) |
| `y` | Rótulos binários (0=good, 1=bad) |
| `preprocess_encode`, `scaler` | Transformadores ajustados (reutilizáveis em `Pipeline`) |

**Próximos passos (futuras seções do notebook):**

- Montar `Pipeline` com pré-processamento + classificador
- Comparar MQ, kNN, MLP, SVM não-linear, AdaBoost e Random Forest
- Avaliar com **validação cruzada $k$-fold** estratificada (conforme a aula)
- Discutir desbalanceamento (ex.: `class_weight`, métricas, eventual SMOTE)


In [ ]:
print("=== Resumo ===")
print(f"Instâncias (n)              : {len(y)}")
print(f"Parâmetros originais (l)    : {X_raw.shape[1]}")
print(f"Parâmetros após one-hot (l'): {X_encoded.shape[1]}")
print(f"Classes — good (0)          : {(y == 0).sum()}")
print(f"Classes — bad  (1)          : {(y == 1).sum()}")
print(f"Relação n/l' após encoding  : {len(y) / X_encoded.shape[1]:.1f}")
